# Data analysis

Calculating accuracies, analysing translations, etc.

In [2]:
import pandas as pd

## Read annotations

In [73]:
a0 = pd.read_excel('data/annotations/a0.xlsm', sheet_name='Sentences', index_col='SENTENCE')
display(a0)

/Users/pivanovs/.pyenv/versions/mt-proj/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,ID,GENDER,OTHER,COMMENT
SENTENCE,,,,
Я працював барменом.,0,male,NaN,NaN
Я бідний бармен.,1,male,NaN,NaN
Я хороший бармен.,2,male,NaN,NaN
Бармен зателефонував мені сьогодні.,3,male,NaN,NaN
Анна - бармен.,4,male,NaN,NaN
...,...,...,...,...
Джессі - добрий воїн.,2764,male,NaN,NaN
Джессі - бідний вояк.,2765,male,NaN,NaN
Анна агентка.,2766,female,NaN,NaN


In [74]:
a1 = pd.read_excel('data/annotations/a1.xlsm', sheet_name='Sentences', index_col='SENTENCE')
display(a1)

,Unnamed: 0,GENDER,OTHER,COMMENT
SENTENCE,,,,
Я працював барменом.,0,male,NaN,NaN
Я бідний бармен.,1,male,NaN,NaN
Я хороший бармен.,2,male,NaN,NaN
Бармен зателефонував мені сьогодні.,3,male,NaN,NaN
Анна - бармен.,4,male,NaN,NaN
...,...,...,...,...
Джессі - добрий воїн.,2764,male,NaN,NaN
Джессі - бідний вояк.,2765,male,NaN,NaN
Анна агентка.,2766,female,NaN,NaN


## Combine and save annotations to a file

In [75]:
annotations = a0.join(a1, on='SENTENCE', lsuffix='_A0', rsuffix='_A1')
# annotations.rename(columns={})
annotations.drop(columns=['ID', 'OTHER_A0', 'OTHER_A1', 'COMMENT_A0', 'COMMENT_A1', 'Unnamed: 0'], inplace=True)
# annotations.set_index('SENTENCE', inplace=True)
# annotations.to_csv('data/annotations/annotations_combined.tsv', sep='\t')
display(annotations)

,GENDER_A0,GENDER_A1
SENTENCE,,
Я працював барменом.,male,male
Я бідний бармен.,male,male
Я хороший бармен.,male,male
Бармен зателефонував мені сьогодні.,male,male
Анна - бармен.,male,male
...,...,...
Джессі - добрий воїн.,male,male
Джессі - бідний вояк.,male,male
Анна агентка.,female,female


In [76]:
# annotations.set_index('SENTENCE')
annotations['annotation_match'] = annotations['GENDER_A0'] == annotations['GENDER_A1']
annotations.annotation_match.mean()

np.float64(0.9758035391838209)

### Inter-annotator agreement

In [78]:
from sklearn.metrics import cohen_kappa_score

print(cohen_kappa_score(annotations.GENDER_A0.values, annotations.GENDER_A1.values))

0.9167906802763194


## Calculating accuracies

In [176]:
# translations = pd.read_csv('data/translations.tsv', sep='\t', index_col=None)
# translations = translations.join(annotations, on='target')
# translations.drop(columns=['target_gender'], inplace=True)
# translations.to_csv('translations_annotated.tsv', sep='\t', index=False)

translations = pd.read_csv('translations_annotated.tsv', sep='\t', index_col=None)
display(translations)

,template_type,occupation,source_gender,source,gold,model,pipeline,target,GENDER_A0,GENDER_A1,annotation_match
0,1,bartender,neutral,I used to work as a bartender.,Я працював барменом.,mbart,zeroshot,Я працював барменом.,male,male,True
1,2,bartender,neutral,I am a poor bartender.,Я бідний бармен.,mbart,zeroshot,Я бідний бармен.,male,male,True
2,3,bartender,neutral,I am a nice bartender.,Я хороший бармен.,mbart,zeroshot,Я хороший бармен.,male,male,True
3,4,bartender,neutral,The bartender called me today.,Бармен зателефонував мені сьогодні.,mbart,zeroshot,Бармен зателефонував мені сьогодні.,male,male,True
4,5,bartender,female,Anne is a bartender.,Анна - барменка.,mbart,zeroshot,Анна - бармен.,male,male,True
...,...,...,...,...,...,...,...,...,...,...,...
4975,6,cook,female,Anne is an experienced cook.,Анна - досвідчена кухарка.,opusmt,llm,Анна - досвідчена кухарка.,female,female,True
4976,7,cook,male,Tom is a cook.,Том - кухар.,opusmt,llm,Том кухар.,male,male,True
4977,8,cook,male,Tom is a poor cook.,Том - бідний кухар.,opusmt,llm,Том - поганий кухар.,male,male,True
4978,9,cook,neutral,Jessie is a good cook.,Джессі - хороший кухар.,opusmt,llm,Джессі добре готує.,none,male,False


In [178]:
translations_normalized = translations.replace({'source_gender': {'neutral': 'male'}})

In [177]:
def get_gender_accuracy(df: pd.DataFrame, col1: str, col2: str):
    matches = df[col1] == df[col2]
    return matches.value_counts(normalize=True)[True]

In [123]:
splits = [split for split in translations.groupby(['model', 'pipeline'])]

for split, df in splits:
    acc = get_gender_accuracy(df, 'source_gender', 'GENDER_A0')
    print(f'Pipeline {split} accuracy: {acc:.2%}')

Pipeline ('mbart', 'llm') accuracy: 34.6%
Pipeline ('mbart', 'tagged') accuracy: 20.4%
Pipeline ('mbart', 'zeroshot') accuracy: 28.8%
Pipeline ('opusmt', 'llm') accuracy: 32.8%
Pipeline ('opusmt', 'tagged') accuracy: 21.4%
Pipeline ('opusmt', 'zeroshot') accuracy: 23.5%


In [180]:
splits = [split for split in translations_normalized.groupby(['model', 'pipeline'])]

for split, df in splits:
    acc = get_gender_accuracy(df, 'source_gender', 'GENDER_A0')
    print(f'Pipeline {split} accuracy: {acc:.2%}')

Pipeline ('mbart', 'llm') accuracy: 86.87%
Pipeline ('mbart', 'tagged') accuracy: 77.47%
Pipeline ('mbart', 'zeroshot') accuracy: 81.81%
Pipeline ('opusmt', 'llm') accuracy: 84.58%
Pipeline ('opusmt', 'tagged') accuracy: 77.59%
Pipeline ('opusmt', 'zeroshot') accuracy: 76.39%


## Case analysis

In [201]:
keys = ['template_type', 'occupation', 'model']
old = translations_normalized[translations_normalized.pipeline == 'zeroshot']
print(len(old))
new = translations_normalized[translations_normalized.pipeline == 'llm']

old = old.drop_duplicates(subset=keys, keep='first')
new = new.drop_duplicates(subset=keys, keep='first')

merge = pd.merge(old, new, on=['template_type', 'occupation', 'source_gender', 'source', 'gold', 'model'], how='left', suffixes=['_old', '_new'])
merge.drop(columns=['GENDER_A1_new', 'GENDER_A1_old', 'annotation_match_new', 'annotation_match_old', 'gold'], inplace=True)

female = merge.query("template_type in (5, 6)")

1660


In [189]:
merge[merge.GENDER_A0_gt == 'female']

,template_type,occupation,source_gender,source,gold,model,pipeline_zs,target_zs,GENDER_A0_zs,GENDER_A1_zs,annotation_match_zs,pipeline_gt,target_gt,GENDER_A0_gt,GENDER_A1_gt,annotation_match_gt
13,4,paramedic,male,The paramedic called me today.,Парамедик зателефонував мені сьогодні.,mbart,zeroshot,Сьогодні мені подзвонив парамедик.,male,male,True,tagged,Медсестра зателефонував мені сьогодні.,female,female,True
194,5,dietitian,female,Anne is a dietitian.,Анна - дієтологиня.,mbart,zeroshot,Анна - дієтолог.,male,male,True,tagged,Анна — дієта.,female,male,False
321,2,secretary,male,I am a poor secretary.,Я бідний секретар.,mbart,zeroshot,Я бідна секретарка.,female,female,True,tagged,Я бідна секретарка.,female,female,True
322,3,secretary,male,I am a nice secretary.,Я хороший секретар.,mbart,zeroshot,Я гарна секретарка.,female,female,True,tagged,Я гарна секретарка.,female,female,True
323,4,secretary,male,The secretary called me today.,Секретар зателефонував мені сьогодні.,mbart,zeroshot,Sekretарка зателефонувала мені сьогодні.,female,female,True,tagged,Секретарка зателефонувала мені сьогодні.,female,female,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1449,10,beautician,male,Jessie is a poor beautician.,Джессі - бідний косметолог.,opusmt,zeroshot,Джессі - бідний красунчик.,male,male,True,tagged,Джессі бідолаха.,female,female,True
1510,1,assistant,male,I used to work as a assistant.,Я працював асистентом.,opusmt,zeroshot,Я колись працювала асистентом.,female,female,True,tagged,Я колись працював помічника.,female,male,False
1514,5,assistant,female,Anne is a assistant.,Анна - асистентка.,opusmt,zeroshot,Анна - асистент.,male,male,True,tagged,Аня помічниця.,female,female,True
1544,5,stylist,female,Anne is a stylist.,Анна - стилістка.,opusmt,zeroshot,Анна - стиліст.,male,male,True,tagged,Аня стиліста.,female,female,True
